# Unmet Need Feature Engineering (MVP)

## Purpose

This notebook builds the first MVP version of the Unmet Need Signal Framework.

The objective is to identify specialties that exhibit combinations of:

- Poor outcomes (Severity)
- NHS service pressure (Pressure)
- Deprivation-related inequalities (Inequalities)

The framework is intended to support Horizon Scanning and Demand Signalling by
highlighting specialties that may warrant further investigation for research
and innovation opportunities.

The framework does NOT:

- Rank ICBs
- Produce a definitive unmet need score
- Determine priority specialties automatically

Instead it produces transparent evidence-based signals to support discussion
and decision-making.

### Current MVP Scope

Included:

- Mortality
- RTT
- Elective length of stay
- Emergency length of stay
- IMD

Retained as contextual information:

- Population
- Geography

Not currently used for prioritisation:

- Prevalence indicators

Those may be incorporated in a future version once a defensible method of
aggregation has been agreed.

## Key Principle

This MVP identifies signals consistent with potential unmet need.

It does not identify research priorities directly.

The intended workflow is:

ICB Data
↓
Severity Signals
↓
Pressure Signals
↓
Inequality Signals
↓
Potential Unmet Need Signals
↓
Further Investigation
↓
Research & Innovation Opportunity Assessment

In [15]:
from pathlib import Path

import pandas as pd
import numpy as np

In [16]:
from pathlib import Path

INPUT_DIR = Path("../Inputs")
OUTPUT_DIR = Path("../Outputs")

FEATURE_ENGINEERING_OUTPUT_DIR = (
    OUTPUT_DIR
    / "Feature_Engineering"
)

FEATURE_ENGINEERING_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXPECTED_ICBS = 42

In [17]:
SPECIALTY_MASTER_FILE = (
    OUTPUT_DIR
    / "Specialty_Master.xlsx"
)

print(
    SPECIALTY_MASTER_FILE
)

..\Outputs\Specialty_Master.xlsx


In [18]:
specialty_master = pd.read_excel(
    SPECIALTY_MASTER_FILE
)

print(
    f"Rows loaded: {len(specialty_master)}"
)

print(
    f"Columns loaded: "
    f"{len(specialty_master.columns)}"
)

display(
    specialty_master.head()
)

Rows loaded: 42
Columns loaded: 19


,ICB_Geography_Code,ICB_Code,ICB_Name,Population_2024,IMD_Average_Score,Under75_CVD_Mortality,Cardiology_RTT_18plus,Cardiology_ElectiveAdmissions_20day+,Cardiology_EmergencyAdmissions_20day+,Heart_Failure_Prevalence,Hypertension_Prevalence,CoronaryHeartDisease_Prevalence,PeripheralArterialDisease_Prevalence,Under75_Respiratory_Mortality,Respiratory_RTT_18plus,Respiratory_ElectiveAdmissions_20day+,Respiratory_EmergencyAdmissions_20day+,COPD_Prevalence,Asthma_Prevalence
0,E54000040,QOX,"Bath And North East Somerset, Swindon And Wilt...",974918,14.281,58.2,43.27,4.3,6.7,1.3,16.0,3.2,0.5,18.0,35.73,14.3,8.1,1.8,7.3
1,E54000024,QHG,"Bedfordshire, Luton And Milton Keynes ICB",1063247,19.537,65.6,47.93,2.8,12.0,0.9,14.1,2.5,0.4,19.0,49.15,NaN,11.4,1.6,6.4
2,E54000055,QHL,Birmingham And Solihull ICB,1404860,34.770,84.2,33.14,0.7,6.5,0.8,13.1,2.6,0.4,18.0,54.67,1.4,14.5,1.5,6.3
3,E54000062,QUA,Black Country ICB,1262719,30.214,96.7,47.46,0.9,2.1,1.1,16.2,3.5,0.5,25.0,18.91,0.0,12.4,2.1,6.6
4,E54000039,QUY,"Bristol, North Somerset And South Gloucestersh...",1025309,17.706,62.0,35.21,4.2,7.6,1.1,13.8,2.8,0.5,17.0,19.83,0.0,7.2,1.9,6.7


In [19]:
EXPECTED_COLUMNS = [
    "ICB_Geography_Code",
    "ICB_Code",
    "ICB_Name",
    "Population_2024",
    "IMD_Average_Score",
    "Under75_CVD_Mortality",
    "Cardiology_RTT_18plus",
    "Cardiology_ElectiveAdmissions_20day+",
    "Cardiology_EmergencyAdmissions_20day+",
    "Under75_Respiratory_Mortality",
    "Respiratory_RTT_18plus",
    "Respiratory_ElectiveAdmissions_20day+",
    "Respiratory_EmergencyAdmissions_20day+"
]

missing_columns = [
    column
    for column in EXPECTED_COLUMNS
    if column not in specialty_master.columns
]

if missing_columns:
    raise ValueError(
        "Missing expected columns:\n"
        f"{missing_columns}"
    )

print(
    "Validation passed."
)

Validation passed.


In [20]:
missing_summary = (
    specialty_master
    .isna()
    .sum()
    .reset_index()
)

missing_summary.columns = [
    "Column",
    "Missing_Values"
]

display(
    missing_summary
    .query("Missing_Values > 0")
)

,Column,Missing_Values
13,Under75_Respiratory_Mortality,2
15,Respiratory_ElectiveAdmissions_20day+,1


In [21]:
specialty_master.to_excel(
    FEATURE_ENGINEERING_OUTPUT_DIR
    / "Input_Snapshot.xlsx",
    index=False
)

In [22]:
mvp_feature_config = pd.DataFrame({
    "Specialty": [
        "Cardiovascular",
        "Cardiovascular",
        "Cardiovascular",
        "Cardiovascular",
        "Respiratory",
        "Respiratory",
        "Respiratory",
        "Respiratory"
    ],

    "Domain": [
        "Severity",
        "Pressure",
        "Pressure",
        "Pressure",
        "Severity",
        "Pressure",
        "Pressure",
        "Pressure"
    ],

    "Measure_Family": [
        "Under75_Mortality",
        "RTT_18plus",
        "Elective_LOS_20plus",
        "Emergency_LOS_20plus",
        "Under75_Mortality",
        "RTT_18plus",
        "Elective_LOS_20plus",
        "Emergency_LOS_20plus"
    ],

    "Indicator": [
        "Under75_CVD_Mortality",
        "Cardiology_RTT_18plus",
        "Cardiology_ElectiveAdmissions_20day+",
        "Cardiology_EmergencyAdmissions_20day+",
        "Under75_Respiratory_Mortality",
        "Respiratory_RTT_18plus",
        "Respiratory_ElectiveAdmissions_20day+",
        "Respiratory_EmergencyAdmissions_20day+"
    ],

    "Direction": [
        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse"
    ],

    "Unit": [
        "Rate per 100,000 population",
        "Percentage of incomplete RTT pathways waiting over 18 weeks",
        "Percentage of elective admissions with length of stay over 20 days",
        "Percentage of emergency admissions with length of stay over 20 days",
        "Rate per 100,000 population",
        "Percentage of incomplete RTT pathways waiting over 18 weeks",
        "Percentage of elective admissions with length of stay over 20 days",
        "Percentage of emergency admissions with length of stay over 20 days"
    ]
})

display(mvp_feature_config)

,Specialty,Domain,Measure_Family,Indicator,Direction,Unit
0,Cardiovascular,Severity,Under75_Mortality,Under75_CVD_Mortality,Higher_Worse,"Rate per 100,000 population"
1,Cardiovascular,Pressure,RTT_18plus,Cardiology_RTT_18plus,Higher_Worse,Percentage of incomplete RTT pathways waiting ...
2,Cardiovascular,Pressure,Elective_LOS_20plus,Cardiology_ElectiveAdmissions_20day+,Higher_Worse,Percentage of elective admissions with length ...
3,Cardiovascular,Pressure,Emergency_LOS_20plus,Cardiology_EmergencyAdmissions_20day+,Higher_Worse,Percentage of emergency admissions with length...
4,Respiratory,Severity,Under75_Mortality,Under75_Respiratory_Mortality,Higher_Worse,"Rate per 100,000 population"
5,Respiratory,Pressure,RTT_18plus,Respiratory_RTT_18plus,Higher_Worse,Percentage of incomplete RTT pathways waiting ...
6,Respiratory,Pressure,Elective_LOS_20plus,Respiratory_ElectiveAdmissions_20day+,Higher_Worse,Percentage of elective admissions with length ...
7,Respiratory,Pressure,Emergency_LOS_20plus,Respiratory_EmergencyAdmissions_20day+,Higher_Worse,Percentage of emergency admissions with length...


### Validate the MVP configuration

The following checks ensure that every configured MVP indicator exists in the
combined master dataset and that every measure family has comparable indicators.

For the current MVP, each measure family should contain one Cardiovascular
indicator and one Respiratory indicator.

In [23]:
required_mvp_columns = [
    "Specialty",
    "Domain",
    "Measure_Family",
    "Indicator",
    "Direction",
    "Unit"
]

missing_mvp_columns = [
    column
    for column in required_mvp_columns
    if column not in mvp_feature_config.columns
]

if missing_mvp_columns:
    raise ValueError(
        "MVP feature configuration is missing columns: "
        f"{missing_mvp_columns}"
    )


missing_mvp_indicators = [
    indicator
    for indicator in mvp_feature_config["Indicator"]
    if indicator not in specialty_master.columns
]

if missing_mvp_indicators:
    raise ValueError(
        "The following MVP indicators are not present in "
        "specialty_master: "
        f"{missing_mvp_indicators}"
    )


duplicate_mvp_indicators = (
    mvp_feature_config["Indicator"]
    .duplicated(keep=False)
)

if duplicate_mvp_indicators.any():
    raise ValueError(
        "Duplicate indicators found in the MVP configuration:\n"
        f"{mvp_feature_config.loc[
            duplicate_mvp_indicators,
            ['Specialty', 'Indicator']
        ]}"
    )


measure_family_check = (
    mvp_feature_config
    .groupby(
        [
            "Measure_Family",
            "Specialty"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

display(measure_family_check)

Specialty,Cardiovascular,Respiratory
Measure_Family,,
Elective_LOS_20plus,1,1
Emergency_LOS_20plus,1,1
RTT_18plus,1,1
Under75_Mortality,1,1


### Create a long-format clinical signal dataset

The master dataset currently has one row per ICB and one column per indicator.

For feature engineering, the data are converted to long format so that each row
represents one:

- ICB
- specialty
- domain

In [24]:
mvp_indicator_columns = (
    mvp_feature_config["Indicator"]
    .tolist()
)

mvp_long = specialty_master.melt(
    id_vars=[
        "ICB_Geography_Code",
        "ICB_Code",
        "ICB_Name",
        "Population_2024",
        "IMD_Average_Score"
    ],
    value_vars=mvp_indicator_columns,
    var_name="Indicator",
    value_name="Raw_Value"
)

mvp_long = mvp_long.merge(
    mvp_feature_config,
    on="Indicator",
    how="left",
    validate="many_to_one"
)

mvp_long["Raw_Value"] = pd.to_numeric(
    mvp_long["Raw_Value"],
    errors="coerce"
)

mvp_long["Population_2024"] = pd.to_numeric(
    mvp_long["Population_2024"],
    errors="coerce"
)

mvp_long["IMD_Average_Score"] = pd.to_numeric(
    mvp_long["IMD_Average_Score"],
    errors="coerce"
)

display(mvp_long.head())

,ICB_Geography_Code,ICB_Code,ICB_Name,Population_2024,IMD_Average_Score,Indicator,Raw_Value,Specialty,Domain,Measure_Family,Direction,Unit
0,E54000040,QOX,"Bath And North East Somerset, Swindon And Wilt...",974918,14.281,Under75_CVD_Mortality,58.2,Cardiovascular,Severity,Under75_Mortality,Higher_Worse,"Rate per 100,000 population"
1,E54000024,QHG,"Bedfordshire, Luton And Milton Keynes ICB",1063247,19.537,Under75_CVD_Mortality,65.6,Cardiovascular,Severity,Under75_Mortality,Higher_Worse,"Rate per 100,000 population"
2,E54000055,QHL,Birmingham And Solihull ICB,1404860,34.770,Under75_CVD_Mortality,84.2,Cardiovascular,Severity,Under75_Mortality,Higher_Worse,"Rate per 100,000 population"
3,E54000062,QUA,Black Country ICB,1262719,30.214,Under75_CVD_Mortality,96.7,Cardiovascular,Severity,Under75_Mortality,Higher_Worse,"Rate per 100,000 population"
4,E54000039,QUY,"Bristol, North Somerset And South Gloucestersh...",1025309,17.706,Under75_CVD_Mortality,62.0,Cardiovascular,Severity,Under75_Mortality,Higher_Worse,"Rate per 100,000 population"


### Create relative signal values within matched measure families

Raw mortality, RTT and length-of-stay measures cannot be averaged together
because they have different meanings and scales.

Each value is therefore converted to a relative 0–100 signal within its
matched measure family.

For example, all Cardiovascular and Respiratory RTT observations are pooled
before the RTT percentiles are calculated. This preserves differences between
the specialties while ensuring that only comparable measures are assessed
together.

A value near 100 means that the observation is relatively high compared with
the other ICB-specialty observations for that same measure family.

These are relative analytical signals. They are not external performance
standards and should not be described as good, bad or clinically acceptable.

In [25]:
def create_relative_signal(
    dataframe,
    value_column="Raw_Value"
):
    """
    Create a direction-aware 0-100 relative signal
    within each matched measure family.
    """

    dataframe = dataframe.copy()

    dataframe[
        "Relative_Signal_0_100"
    ] = np.nan

    for measure_family, family_data in (
        dataframe.groupby(
            "Measure_Family"
        )
    ):

        valid_mask = (
            dataframe["Measure_Family"].eq(
                measure_family
            )
            & dataframe[
                value_column
            ].notna()
        )

        direction_values = (
            family_data["Direction"]
            .dropna()
            .unique()
        )

        if len(direction_values) != 1:
            raise ValueError(
                f"Measure family {measure_family} "
                "contains inconsistent Direction values: "
                f"{direction_values}"
            )

        direction = direction_values[0]

        percentile_signal = (
            dataframe.loc[
                valid_mask,
                value_column
            ]
            .rank(
                method="average",
                pct=True
            )
            .mul(100)
        )

        if direction == "Higher_Worse":

            dataframe.loc[
                valid_mask,
                "Relative_Signal_0_100"
            ] = percentile_signal

        elif direction == "Lower_Worse":

            dataframe.loc[
                valid_mask,
                "Relative_Signal_0_100"
            ] = (
                100
                - percentile_signal
            )

        else:
            raise ValueError(
                f"Unexpected Direction value: "
                f"{direction}"
            )

    return dataframe


mvp_long = create_relative_signal(
    dataframe=mvp_long
)

display(
    mvp_long[
        [
            "ICB_Name",
            "Specialty",
            "Domain",
            "Measure_Family",
            "Raw_Value",
            "Relative_Signal_0_100"
        ]
    ].head(10)
)

,ICB_Name,Specialty,Domain,Measure_Family,Raw_Value,Relative_Signal_0_100
0,"Bath And North East Somerset, Swindon And Wilt...",Cardiovascular,Severity,Under75_Mortality,58.2,56.097561
1,"Bedfordshire, Luton And Milton Keynes ICB",Cardiovascular,Severity,Under75_Mortality,65.6,74.390244
2,Birmingham And Solihull ICB,Cardiovascular,Severity,Under75_Mortality,84.2,94.512195
3,Black Country ICB,Cardiovascular,Severity,Under75_Mortality,96.7,100.000000
4,"Bristol, North Somerset And South Gloucestersh...",Cardiovascular,Severity,Under75_Mortality,62.0,67.073171
5,"Buckinghamshire, Oxfordshire And Berkshire Wes...",Cardiovascular,Severity,Under75_Mortality,54.3,51.219512
6,Cambridgeshire And Peterborough ICB,Cardiovascular,Severity,Under75_Mortality,52.7,50.000000
7,Cheshire And Merseyside ICB,Cardiovascular,Severity,Under75_Mortality,83.7,92.682927
8,Cornwall And The Isles Of Scilly ICB,Cardiovascular,Severity,Under75_Mortality,66.8,76.829268
9,Coventry And Warwickshire ICB,Cardiovascular,Severity,Under75_Mortality,81.0,90.243902


### Summarise each matched indicator family by specialty

This stage creates one row for each specialty and measure family.

The table retains:

- raw median value;
- relative signal value;
- data coverage;
- geographic variation;
- association with deprivation.

The raw median is retained because the 0–100 relative signal does not show the
original magnitude of the measure.

The coefficient of variation and IMD correlation are supporting analytical
features. They are not included in the core severity or pressure score.

In [26]:
def safe_correlation(
    dataframe,
    value_column,
    context_column,
    minimum_pairs=10
):
    paired_data = dataframe[
        [
            value_column,
            context_column
        ]
    ].dropna()

    if len(paired_data) < minimum_pairs:
        return np.nan

    if (
        paired_data[
            value_column
        ].nunique() < 2
        or paired_data[
            context_column
        ].nunique() < 2
    ):
        return np.nan

    return paired_data[
        value_column
    ].corr(
        paired_data[
            context_column
        ],
        method="pearson"
    )


indicator_family_records = []

for (
    specialty,
    domain,
    measure_family,
    indicator
), indicator_data in mvp_long.groupby(
    [
        "Specialty",
        "Domain",
        "Measure_Family",
        "Indicator"
    ]
):

    valid_values = indicator_data[
        "Raw_Value"
    ].dropna()

    valid_signals = indicator_data[
        "Relative_Signal_0_100"
    ].dropna()

    if valid_values.empty:
        continue

    raw_mean = valid_values.mean()
    raw_std = valid_values.std()

    if raw_mean != 0:
        coefficient_of_variation = (
            raw_std
            / abs(raw_mean)
        )
    else:
        coefficient_of_variation = np.nan

    indicator_family_records.append({
        "Specialty":
            specialty,
        "Domain":
            domain,
        "Measure_Family":
            measure_family,
        "Indicator":
            indicator,
        "Unit":
            indicator_data[
                "Unit"
            ].iloc[0],
        "ICBs_Available":
            valid_values.count(),
        "Data_Coverage_Percent":
            (
                valid_values.count()
                / EXPECTED_ICBS
                * 100
            ),
        "Raw_ICB_Mean":
            raw_mean,
        "Raw_ICB_Median":
            valid_values.median(),
        "Raw_ICB_Minimum":
            valid_values.min(),
        "Raw_ICB_Maximum":
            valid_values.max(),
        "Relative_Signal_Mean":
            valid_signals.mean(),
        "Relative_Signal_Median":
            valid_signals.median(),
        "Geographic_Variation_CV":
            coefficient_of_variation,
        "IMD_Correlation":
            safe_correlation(
                dataframe=indicator_data,
                value_column="Raw_Value",
                context_column=(
                    "IMD_Average_Score"
                )
            ),
        "Population_Correlation":
            safe_correlation(
                dataframe=indicator_data,
                value_column="Raw_Value",
                context_column=(
                    "Population_2024"
                )
            )
    })


indicator_family_summary = pd.DataFrame(
    indicator_family_records
)

indicator_family_summary = (
    indicator_family_summary
    .sort_values(
        [
            "Specialty",
            "Domain",
            "Measure_Family"
        ]
    )
    .reset_index(drop=True)
)

display(
    indicator_family_summary.round(3)
)

,Specialty,Domain,Measure_Family,Indicator,Unit,ICBs_Available,Data_Coverage_Percent,Raw_ICB_Mean,Raw_ICB_Median,Raw_ICB_Minimum,Raw_ICB_Maximum,Relative_Signal_Mean,Relative_Signal_Median,Geographic_Variation_CV,IMD_Correlation,Population_Correlation
0,Cardiovascular,Pressure,Elective_LOS_20plus,Cardiology_ElectiveAdmissions_20day+,Percentage of elective admissions with length ...,42,100.000,3.050,2.150,0.00,14.30,56.770,59.337,1.156,-0.095,-0.271
1,Cardiovascular,Pressure,Emergency_LOS_20plus,Cardiology_EmergencyAdmissions_20day+,Percentage of emergency admissions with length...,42,100.000,5.729,5.550,0.00,13.50,36.565,33.333,0.495,0.021,0.146
2,Cardiovascular,Pressure,RTT_18plus,Cardiology_RTT_18plus,Percentage of incomplete RTT pathways waiting ...,42,100.000,37.521,38.600,16.74,53.03,54.833,57.738,0.238,-0.087,-0.008
3,Cardiovascular,Severity,Under75_Mortality,Under75_CVD_Mortality,"Rate per 100,000 population",42,100.000,69.660,65.600,52.70,96.70,75.000,74.390,0.165,0.824,0.392
4,Respiratory,Pressure,Elective_LOS_20plus,Respiratory_ElectiveAdmissions_20day+,Percentage of elective admissions with length ...,41,97.619,2.707,0.000,0.00,20.00,44.284,21.687,1.721,0.034,0.013
5,Respiratory,Pressure,Emergency_LOS_20plus,Respiratory_EmergencyAdmissions_20day+,Percentage of emergency admissions with length...,42,100.000,9.755,9.650,0.00,19.00,64.626,71.429,0.502,0.166,0.266
6,Respiratory,Pressure,RTT_18plus,Respiratory_RTT_18plus,Percentage of incomplete RTT pathways waiting ...,42,100.000,35.012,36.555,13.14,59.35,46.358,47.024,0.289,0.040,-0.111
7,Respiratory,Severity,Under75_Mortality,Under75_Respiratory_Mortality,"Rate per 100,000 population",40,95.238,19.525,20.000,2.00,31.00,25.000,23.780,0.291,0.584,0.284


### Create specialty-level severity and pressure signals

The relative indicator-family signals are now aggregated within each specialty
and domain.

Each matched measure family contributes one value:

- Severity currently contains one measure family: under-75 mortality.
- Pressure currently contains three measure families: RTT, elective long stay
  and emergency long stay.

The mean is taken across measure families, not across raw indicator values.

Because Cardiovascular and Respiratory currently have the same measure
families, neither specialty receives a higher score simply because more
indicators are available.

The number of measure families and coverage are retained so that evidence
completeness remains visible.

In [27]:
specialty_domain_signals = (
    indicator_family_summary
    .groupby(
        [
            "Specialty",
            "Domain"
        ],
        as_index=False
    )
    .agg(
        Domain_Signal_0_100=(
            "Relative_Signal_Median",
            "mean"
        ),
        Number_of_Measure_Families=(
            "Measure_Family",
            "nunique"
        ),
        Median_Data_Coverage_Percent=(
            "Data_Coverage_Percent",
            "median"
        ),
        Median_Geographic_Variation_CV=(
            "Geographic_Variation_CV",
            "median"
        ),
        Mean_IMD_Correlation=(
            "IMD_Correlation",
            "mean"
        ),
        Maximum_Absolute_IMD_Correlation=(
            "IMD_Correlation",
            lambda values: (
                values.abs().max()
            )
        )
    )
)

display(
    specialty_domain_signals.round(3)
)

,Specialty,Domain,Domain_Signal_0_100,Number_of_Measure_Families,Median_Data_Coverage_Percent,Median_Geographic_Variation_CV,Mean_IMD_Correlation,Maximum_Absolute_IMD_Correlation
0,Cardiovascular,Pressure,50.136,3,100.000,0.495,-0.054,0.095
1,Cardiovascular,Severity,74.390,1,100.000,0.165,0.824,0.824
2,Respiratory,Pressure,46.713,3,100.000,0.502,0.080,0.166
3,Respiratory,Severity,23.780,1,95.238,0.291,0.584,0.584


### Create the inequalities profile
 
Inequalities are not treated as another clinical indicator.
 
Instead, the framework examines whether severity and pressure indicators tend
to be higher in ICBs with higher average IMD scores.
 
A positive correlation means that the clinical indicator tends to be higher
in more deprived ICBs.
 
A negative correlation means that the indicator tends to be lower in more
deprived ICBs.
 
A value near zero means that there is little linear association at ICB level.
 
These are ecological, area-level associations. They do not demonstrate
individual-level inequalities or causation.

In [28]:
inequality_profile = (
    indicator_family_summary
    .groupby(
        [
            "Specialty",
            "Domain"
        ],
        as_index=False
    )
    .agg(
        Mean_IMD_Association=(
            "IMD_Correlation",
            "mean"
        ),
        Median_IMD_Association=(
            "IMD_Correlation",
            "median"
        ),
        Strongest_Absolute_IMD_Association=(
            "IMD_Correlation",
            lambda values: (
                values.abs().max()
            )
        ),
        Indicators_With_IMD_Result=(
            "IMD_Correlation",
            "count"
        )
    )
)

display(
    inequality_profile.round(3)
)

,Specialty,Domain,Mean_IMD_Association,Median_IMD_Association,Strongest_Absolute_IMD_Association,Indicators_With_IMD_Result
0,Cardiovascular,Pressure,-0.054,-0.087,0.095,3
1,Cardiovascular,Severity,0.824,0.824,0.824,1
2,Respiratory,Pressure,0.080,0.040,0.166,3
3,Respiratory,Severity,0.584,0.584,0.584,1


### Create the geographic variation profile

Geographic variation is retained as supporting information rather than being
included in the core severity or pressure signal.

Higher variation may indicate that the specialty signal is unevenly
distributed across England.

This can help generate hypotheses about differences in access, pathways,
workforce, coding, case mix, service configuration or adoption of effective
practice.

Variation alone does not demonstrate unmet need or an innovation gap.

In [29]:
variation_profile = (
    indicator_family_summary
    .groupby(
        [
            "Specialty",
            "Domain"
        ],
        as_index=False
    )
    .agg(
        Median_Variation_CV=(
            "Geographic_Variation_CV",
            "median"
        ),
        Maximum_Variation_CV=(
            "Geographic_Variation_CV",
            "max"
        ),
        Number_of_Indicators=(
            "Indicator",
            "nunique"
        )
    )
)

display(
    variation_profile.round(3)
)

,Specialty,Domain,Median_Variation_CV,Maximum_Variation_CV,Number_of_Indicators
0,Cardiovascular,Pressure,0.495,1.156,3
1,Cardiovascular,Severity,0.165,0.165,1
2,Respiratory,Pressure,0.502,1.721,3
3,Respiratory,Severity,0.291,0.291,1


### Create the continuous Specialty Signal Summary
 
The final MVP output contains one row per specialty.
 
It presents:
 
- a relative severity signal;
- a relative pressure signal;
- severity and pressure associations with deprivation;
- severity and pressure geographic variation;
- evidence completeness.
 
No overall score is calculated at this stage.
 
The output is intended to identify specialties exhibiting combinations of
signals that may justify deeper R&I investigation. It is not a definitive
priority ranking.

In [30]:
# Core severity and pressure signals

core_signal_wide = (
    specialty_domain_signals
    .pivot(
        index="Specialty",
        columns="Domain",
        values="Domain_Signal_0_100"
    )
    .rename(
        columns={
            "Severity":
                "Severity_Signal_0_100",
            "Pressure":
                "Pressure_Signal_0_100"
        }
    )
    .reset_index()
)


# Inequality associations by domain

inequality_wide = (
    inequality_profile
    .pivot(
        index="Specialty",
        columns="Domain",
        values="Mean_IMD_Association"
    )
    .rename(
        columns={
            "Severity":
                "Severity_IMD_Association",
            "Pressure":
                "Pressure_IMD_Association"
        }
    )
    .reset_index()
)


# Geographic variation by domain

variation_wide = (
    variation_profile
    .pivot(
        index="Specialty",
        columns="Domain",
        values="Median_Variation_CV"
    )
    .rename(
        columns={
            "Severity":
                "Severity_Variation_CV",
            "Pressure":
                "Pressure_Variation_CV"
        }
    )
    .reset_index()
)


# Indicator / measure-family counts

evidence_summary = (
    specialty_domain_signals
    .pivot(
        index="Specialty",
        columns="Domain",
        values="Number_of_Measure_Families"
    )
    .rename(
        columns={
            "Severity":
                "Severity_Measure_Families",
            "Pressure":
                "Pressure_Measure_Families"
        }
    )
    .reset_index()
)


# Combine the profiles

specialty_signal_summary = (
    core_signal_wide
    .merge(
        inequality_wide,
        on="Specialty",
        how="outer",
        validate="one_to_one"
    )
    .merge(
        variation_wide,
        on="Specialty",
        how="outer",
        validate="one_to_one"
    )
    .merge(
        evidence_summary,
        on="Specialty",
        how="outer",
        validate="one_to_one"
    )
)


# Put columns into a logical order

final_columns = [
    "Specialty",
    "Severity_Signal_0_100",
    "Pressure_Signal_0_100",
    "Severity_IMD_Association",
    "Pressure_IMD_Association",
    "Severity_Variation_CV",
    "Pressure_Variation_CV",
    "Severity_Measure_Families",
    "Pressure_Measure_Families"
]

for column in final_columns:
    if column not in specialty_signal_summary.columns:
        specialty_signal_summary[column] = np.nan

specialty_signal_summary = (
    specialty_signal_summary[
        final_columns
    ]
    .sort_values(
        "Specialty"
    )
    .reset_index(drop=True)
)

display(
    specialty_signal_summary.round(3)
)

Domain,Specialty,Severity_Signal_0_100,Pressure_Signal_0_100,Severity_IMD_Association,Pressure_IMD_Association,Severity_Variation_CV,Pressure_Variation_CV,Severity_Measure_Families,Pressure_Measure_Families
0,Cardiovascular,74.39,50.136,0.824,-0.054,0.165,0.495,1,3
1,Respiratory,23.78,46.713,0.584,0.080,0.291,0.502,1,3


### Create a dashboard-ready ICB diagnostic dataset

The specialty summary identifies broad patterns at specialty level.

The ICB diagnostic dataset retains the geographical detail needed for maps and
drill-down analysis.

Population is included here as local context. It is not treated as the number
of people affected by the selected specialty.

This dataset supports questions such as:

- Where is severity relatively high?
- Where is pressure relatively high?
- Do high relative signals appear in more deprived ICBs?
- Are signals geographically concentrated?

In [31]:
dashboard_icb_signals = (
    mvp_long[
        [
            "ICB_Geography_Code",
            "ICB_Code",
            "ICB_Name",
            "Population_2024",
            "IMD_Average_Score",
            "Specialty",
            "Domain",
            "Measure_Family",
            "Indicator",
            "Unit",
            "Raw_Value",
            "Relative_Signal_0_100"
        ]
    ]
    .copy()
)

display(
    dashboard_icb_signals.head()
)

,ICB_Geography_Code,ICB_Code,ICB_Name,Population_2024,IMD_Average_Score,Specialty,Domain,Measure_Family,Indicator,Unit,Raw_Value,Relative_Signal_0_100
0,E54000040,QOX,"Bath And North East Somerset, Swindon And Wilt...",974918,14.281,Cardiovascular,Severity,Under75_Mortality,Under75_CVD_Mortality,"Rate per 100,000 population",58.2,56.097561
1,E54000024,QHG,"Bedfordshire, Luton And Milton Keynes ICB",1063247,19.537,Cardiovascular,Severity,Under75_Mortality,Under75_CVD_Mortality,"Rate per 100,000 population",65.6,74.390244
2,E54000055,QHL,Birmingham And Solihull ICB,1404860,34.770,Cardiovascular,Severity,Under75_Mortality,Under75_CVD_Mortality,"Rate per 100,000 population",84.2,94.512195
3,E54000062,QUA,Black Country ICB,1262719,30.214,Cardiovascular,Severity,Under75_Mortality,Under75_CVD_Mortality,"Rate per 100,000 population",96.7,100.000000
4,E54000039,QUY,"Bristol, North Somerset And South Gloucestersh...",1025309,17.706,Cardiovascular,Severity,Under75_Mortality,Under75_CVD_Mortality,"Rate per 100,000 population",62.0,67.073171


### Export MVP unmet need signal outputs

The exported workbook contains the complete audit trail from individual ICB
observations through to the specialty-level summary.

The detailed sheets should be retained alongside any future dashboard so that
the derivation of every signal remains transparent.

In [32]:
with pd.ExcelWriter(
    FEATURE_ENGINEERING_OUTPUT_DIR
    / "MVP_Unmet_Need_Signals.xlsx",
    engine="openpyxl"
) as writer:

    mvp_feature_config.to_excel(
        writer,
        sheet_name="MVP Configuration",
        index=False
    )

    indicator_family_summary.to_excel(
        writer,
        sheet_name="Indicator Signals",
        index=False
    )

    specialty_domain_signals.to_excel(
        writer,
        sheet_name="Domain Signals",
        index=False
    )

    inequality_profile.to_excel(
        writer,
        sheet_name="Inequality Profile",
        index=False
    )

    variation_profile.to_excel(
        writer,
        sheet_name="Variation Profile",
        index=False
    )

    specialty_signal_summary.to_excel(
        writer,
        sheet_name="Specialty Summary",
        index=False
    )

    dashboard_icb_signals.to_excel(
        writer,
        sheet_name="Dashboard ICB Data",
        index=False
    )


print("=" * 60)
print("MVP UNMET NEED SIGNAL FRAMEWORK COMPLETE")
print("=" * 60)

print(
    f"Specialties included: "
    f"{mvp_feature_config['Specialty'].nunique()}"
)

print(
    f"Core indicators included: "
    f"{mvp_feature_config['Indicator'].nunique()}"
)

print(
    f"Measure families included: "
    f"{mvp_feature_config['Measure_Family'].nunique()}"
)

print(
    f"ICBs represented: "
    f"{specialty_master['ICB_Code'].nunique()}"
)

print(
    "Output saved as: "
    "MVP_Unmet_Need_Signals.xlsx"
)

MVP UNMET NEED SIGNAL FRAMEWORK COMPLETE
Specialties included: 2
Core indicators included: 8
Measure families included: 4
ICBs represented: 42
Output saved as: MVP_Unmet_Need_Signals.xlsx
